# CogMem Cognitive Patches — Minimal Experiment

**Goal:** Verify the cognitive patches architecture works.

**Plan:**
1. Load base model (4-bit, ~2GB VRAM)
2. Process first 100 tasks with N=4 candidates per task
3. Create patches from pass/fail contrasts (~20 patches expected)
4. Evaluate patches on remaining 1040 UNSEEN tasks
5. Compare: patched eval > cold eval = architecture works

**Hardware:** A4000 16GB. Model loaded in 4-bit via transformers (not Ollama).
Generation happens through model.generate() directly.


In [ ]:
# Cell 1: Install deps + check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
!pip install transformers accelerate bitsandbytes peft datasets sentence-transformers -q
print("Deps installed")


In [ ]:
# Cell 2: Clone CogMem + load tasks
!cd /notebooks && git clone https://github.com/tungooxx/CogMem.git 2>/dev/null || \
    (cd /notebooks/CogMem && git pull)
!cd /notebooks/CogMem && pip install -e . --no-deps -q

import sys
if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

import json
from pathlib import Path
from datasets import load_dataset

# Load BigCodeBench full (1140 tasks)
TASKS_PATH = "/notebooks/bigcodebench_tasks.jsonl"
if not Path(TASKS_PATH).exists():
    ds = load_dataset("bigcode/bigcodebench", split="v0.1.4")
    tasks = []
    for item in ds:
        tasks.append({
            "task_id": item["task_id"],
            "instruct_prompt": item.get("instruct_prompt", ""),
            "complete_prompt": item.get("complete_prompt", ""),
            "test": item.get("test", ""),
            "entry_point": item.get("entry_point", ""),
        })
    with open(TASKS_PATH, "w") as f:
        for t in tasks:
            f.write(json.dumps(t) + chr(10))
else:
    tasks = []
    with open(TASKS_PATH) as f:
        for line in f:
            if line.strip():
                tasks.append(json.loads(line))

print("Tasks:", len(tasks))
# Split: first 100 for patch creation, rest for evaluation
TRAIN_TASKS = tasks[:100]
EVAL_TASKS = tasks[100:]
print("Train (create patches):", len(TRAIN_TASKS))
print("Eval (test patches):", len(EVAL_TASKS))


In [ ]:
# Cell 3: Load base model (4-bit) + embedder
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("Loading model (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading embedder...")
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

free = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Model loaded. Free VRAM: {free:.1f} GB")
print("Ready for patch creation.")


In [ ]:
# Cell 4: Create patches from first 100 tasks
# Generate N=4 candidates per task, find pass/fail contrasts, create patches
import time
from difflib import SequenceMatcher
from cogmem.benchmarks.bigcodebench.prompts import SYSTEM_PROMPT, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution
from cogmem.patches.patch import CognitivePatch
from cogmem.patches.bank import PatchBank
from cogmem.patches.create import create_patch_from_contrast
from cogmem.patches.wake import generate_with_model, find_best_contrast_pair

N_CANDIDATES = 4
PATCH_DIR = "/notebooks/cogmem_patches"
patch_bank = PatchBank(PATCH_DIR)

total_patches = 0
total_passed = 0
start_time = time.time()

for i, task in enumerate(TRAIN_TASKS):
    task_id = task["task_id"]
    prompt = task.get("instruct_prompt", task.get("complete_prompt", ""))
    task_embedding = embedder.encode(prompt).tolist()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    # Generate N candidates
    candidates = []
    for _ in range(N_CANDIDATES):
        try:
            response = generate_with_model(
                base_model, tokenizer, messages, temperature=0.8
            )
            code = extract_code(response)
            if code and len(code.strip()) > 20:
                result = evaluate_solution(task, code, timeout=30, mode="subprocess")
                candidates.append({"code": code, "passed": result["passed"]})
        except Exception:
            pass

    passes = [c for c in candidates if c["passed"]]
    fails = [c for c in candidates if not c["passed"]]

    if passes:
        total_passed += 1

    # Create patch if we have pass + fail
    if passes and fails:
        best_pair, sim = find_best_contrast_pair(passes, fails)
        if best_pair:
            try:
                patch = create_patch_from_contrast(
                    base_model, tokenizer, prompt,
                    best_pair["fail"]["code"],
                    best_pair["pass"]["code"],
                    patch_id="patch_{}_{}".format(task_id.replace("/", "_"), int(time.time())),
                    rank=2, n_steps=5, lr=1e-3,
                )
                patch.embedding = task_embedding
                patch.source_task_id = task_id
                patch_bank.add(patch)
                total_patches += 1
            except Exception as e:
                print("  Patch creation failed:", str(e)[:80])

    if (i + 1) % 10 == 0 or i < 5:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed * 3600 if elapsed > 0 else 0
        print("[{}/{}] {}: {}P/{}F | patches={} | pass_rate={}/{} | {:.0f}/hr".format(
            i + 1, len(TRAIN_TASKS), task_id,
            len(passes), len(fails), total_patches,
            total_passed, i + 1, rate))

patch_bank.save()
elapsed = (time.time() - start_time) / 60
print()
print("=" * 50)
print("PATCH CREATION COMPLETE")
print("Tasks processed:", len(TRAIN_TASKS))
print("Tasks with passes:", total_passed)
print("Patches created:", total_patches)
print("Time:", round(elapsed, 1), "min")
print("Bank stats:", patch_bank.stats())


In [ ]:
# Cell 5: Evaluate patches on UNSEEN tasks (1040 tasks)
# Compare: cold (no patches) vs patched (with patches)
# Only eval a subset for speed — change EVAL_SIZE for full eval

EVAL_SIZE = 200  # first 200 of the 1040 unseen tasks
eval_subset = EVAL_TASKS[:EVAL_SIZE]
print("Evaluating on", EVAL_SIZE, "unseen tasks")
print("Patches in bank:", len(patch_bank.patches))

# --- Cold eval (no patches) ---
print()
print("--- COLD EVAL (base model, no patches) ---")
cold_passed = 0
for i, task in enumerate(eval_subset):
    prompt = task.get("instruct_prompt", task.get("complete_prompt", ""))
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    try:
        response = generate_with_model(base_model, tokenizer, messages, temperature=0)
        code = extract_code(response)
        result = evaluate_solution(task, code, timeout=30, mode="subprocess")
        if result["passed"]:
            cold_passed += 1
    except Exception:
        pass

    if (i + 1) % 50 == 0:
        print("  [{}/{}] cold: {}/{} ({:.1%})".format(
            i + 1, EVAL_SIZE, cold_passed, i + 1, cold_passed / (i + 1)))

cold_rate = cold_passed / max(EVAL_SIZE, 1)
print("Cold result:", cold_passed, "/", EVAL_SIZE, "({:.1%})".format(cold_rate))

# --- Patched eval (with cognitive patches) ---
print()
print("--- PATCHED EVAL (base model + patches per task) ---")
from cogmem.patches.compose import PatchedModel

patched_passed = 0
for i, task in enumerate(eval_subset):
    prompt = task.get("instruct_prompt", task.get("complete_prompt", ""))
    task_embedding = embedder.encode(prompt).tolist()

    # Get relevant patches for this task
    active_patches = patch_bank.get_active_patches(task_embedding, top_k=5)
    for p in active_patches:
        patch_bank.load_weights(p)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    try:
        with PatchedModel(base_model, active_patches):
            response = generate_with_model(base_model, tokenizer, messages, temperature=0)
        code = extract_code(response)
        result = evaluate_solution(task, code, timeout=30, mode="subprocess")
        if result["passed"]:
            patched_passed += 1
    except Exception:
        pass

    if (i + 1) % 50 == 0:
        print("  [{}/{}] patched: {}/{} ({:.1%})".format(
            i + 1, EVAL_SIZE, patched_passed, i + 1, patched_passed / (i + 1)))

patched_rate = patched_passed / max(EVAL_SIZE, 1)
print("Patched result:", patched_passed, "/", EVAL_SIZE, "({:.1%})".format(patched_rate))


In [ ]:
# Cell 6: Results comparison
print("=" * 50)
print("COGNITIVE PATCHES EXPERIMENT RESULTS")
print("=" * 50)
print()
print("Patches created from first 100 tasks:", len(patch_bank.patches))
print("Evaluated on", EVAL_SIZE, "UNSEEN tasks (tasks 100-{})".format(100 + EVAL_SIZE))
print()
print("{:<20} {:>8} {:>8} {:>10}".format("Model", "Passed", "Total", "Rate"))
print("-" * 48)
print("{:<20} {:>8} {:>8} {:>9.1%}".format("Cold (no patches)", cold_passed, EVAL_SIZE, cold_rate))
print("{:<20} {:>8} {:>8} {:>9.1%}".format("Patched", patched_passed, EVAL_SIZE, patched_rate))
print()
diff = patched_rate - cold_rate
if diff > 0.01:
    print("Patches IMPROVED by {:.1%} on unseen tasks!".format(diff))
    print("The architecture WORKS - patches transfer to new tasks.")
elif diff > -0.01:
    print("No significant difference.")
    print("Patches didn't help (yet). May need more patches or better contrasts.")
else:
    print("Patches HURT by {:.1%}.".format(abs(diff)))
    print("Composition may be interfering. Check patch quality.")

# Patch bank details
print()
print("Patch bank:")
stats = patch_bank.stats()
for k, v in stats.items():
    print("  {}: {}".format(k, v))


In [ ]:
# Cell 7: Inspect individual patches
# See what the patches learned
for patch in patch_bank.patches[:5]:
    print("Patch:", patch.patch_id)
    print("  Source:", patch.source_task_id)
    print("  Type:", patch.source_type)
    print("  Q:", patch.q_value, "visits:", patch.q_visits)
    print("  Description:", patch.description[:100])
    print("  Memory:", patch.memory_bytes(), "bytes")
    print("  Layers:", len(patch.lora_weights))
    print()
